# 03 — Streaming Fundamentals: Producer/Consumer, Backpressure, Partitioning, Offsets

`01-why-distributed-processing` and `02-pyspark-local-mode` both processed a **fixed, already
collected** dataset — batch processing. This notebook asks what changes when data instead arrives
continuously, forever, and has to be processed as it comes in.

Everything below is real standard-library Python, actually executed, with real pasted output. No
broker, no external service — that is deliberate: this topic builds the mechanisms a real broker
(Kafka, in `04-kafka`) automates and hardens, from scratch, so that topic isn't "magic."

Three self-contained experiments:

1. **Backpressure** — a bounded `queue.Queue` shows a fast producer genuinely blocking when a
   slow consumer can't keep up.
2. **Partitioning** — a toy partitioned log (list of lists, keyed by hash) with two independent
   consumers tracking their own offsets.
3. **Crash-and-resume** — a consumer crashes mid-stream; a *fresh* consumer object resumes from
   the last committed offset and finishes correctly, with a matching failure-mode demo of what
   goes wrong without a durable offset store.


## Part 1 — Backpressure with a bounded queue

**Setup.** A bounded `queue.Queue(maxsize=3)` sits between a producer thread that tries to push
6 items as fast as possible, and a consumer thread that takes 0.3s to process each item. `put()`
on a full queue blocks until a slot frees up — that block *is* backpressure: the queue itself
forces the producer to slow down to the consumer's pace, instead of growing without bound.

We time every `put()` call directly to show the blocking, not just assert it.


In [1]:
import queue
import threading
import time

bounded_q = queue.Queue(maxsize=3)
put_log = []

def fast_producer():
    for i in range(6):
        t0 = time.perf_counter()
        bounded_q.put(i)  # blocks once the queue is full
        elapsed = time.perf_counter() - t0
        put_log.append((i, elapsed))
        print(f"  producer: put({i}) took {elapsed:.3f}s (queue size now {bounded_q.qsize()})")

def slow_consumer():
    for _ in range(6):
        item = bounded_q.get()
        time.sleep(0.3)  # slow consumer: 0.3s per item
        print(f"  consumer: processed {item}")
        bounded_q.task_done()

producer_thread = threading.Thread(target=fast_producer)
consumer_thread = threading.Thread(target=slow_consumer)

start = time.perf_counter()
producer_thread.start()
consumer_thread.start()
producer_thread.join()
consumer_thread.join()
total = time.perf_counter() - start

print(f"\ntotal wall time: {total:.3f}s")
print("put() timings (item, seconds):", [(i, round(t, 3)) for i, t in put_log])


  producer: put(0) took 0.000s (queue size now 1)
  producer: put(1) took 0.000s (queue size now 2)
  producer: put(2) took 0.000s (queue size now 3)
  producer: put(3) took 0.000s (queue size now 3)


  consumer: processed 0
  producer: put(4) took 0.306s (queue size now 3)


  consumer: processed 1
  producer: put(5) took 0.300s (queue size now 3)


  consumer: processed 2


  consumer: processed 3


  consumer: processed 4


  consumer: processed 5

total wall time: 1.809s
put() timings (item, seconds): [(0, 0.0), (1, 0.0), (2, 0.0), (3, 0.0), (4, 0.306), (5, 0.3)]


**Reading the output.** The first three `put()` calls (items 0, 1, 2) return instantly — the
queue has room. The consumer's first `get()` also happens almost instantly (before its 0.3s
sleep), freeing one slot, so `put(3)` also returns fast. From then on the queue is genuinely full
between every consumer step: `put(4)` and `put(5)` each measurably block for ~0.3s — exactly the
consumer's per-item processing time. The producer cannot outrun the consumer past the buffer size;
it is *forced* to slow down. That forcing is backpressure. Total wall time (~1.8s) is dominated by
the consumer's 6 × 0.3s of work, not the producer — confirming the consumer, not the queue, is now
the bottleneck once the buffer fills.

Compare this to an **unbounded** `queue.Queue()` (no `maxsize`): `put()` never blocks, the
producer races ahead, and the queue grows without limit — the "just batch every N minutes / never
push back" failure mode. See Failure modes in `notes.md`.


## Part 2 — A toy partitioned log with independent consumer offsets

**Setup.** A stream needs to be processed in parallel, but many use cases require **per-key
ordering** (e.g. all of one user's events must be seen in order). The standard solution:
partition the stream by a stable hash of the key, so every event for a given key always lands in
the same partition — order is preserved *within* a partition, and different partitions can be
consumed independently and in parallel.

Below: 4 partitions (plain Python lists, standing in for Kafka's per-partition append-only logs),
a deterministic `partition_for_key` (SHA-256, not Python's salted `hash()`, so the mapping is
stable across runs/processes), and two independent "consumers" each tracking their own offset per
partition.


In [2]:
import hashlib

NUM_PARTITIONS = 4

def partition_for_key(key: str, num_partitions: int = NUM_PARTITIONS) -> int:
    """Deterministic key -> partition mapping (stable hash, not Python's salted hash())."""
    digest = hashlib.sha256(key.encode()).hexdigest()
    return int(digest, 16) % num_partitions

# the "log": one append-only list per partition
log = [[] for _ in range(NUM_PARTITIONS)]

events = [
    ("user-1", "click"), ("user-2", "click"), ("user-1", "purchase"),
    ("user-3", "click"), ("user-1", "logout"), ("user-2", "purchase"),
    ("user-4", "click"), ("user-1", "click"), ("user-3", "purchase"),
    ("user-2", "logout"),
]

for key, event in events:
    p = partition_for_key(key)
    log[p].append((key, event))

for p, messages in enumerate(log):
    print(f"  partition {p}: {messages}")


  partition 0: []
  partition 1: [('user-3', 'click'), ('user-3', 'purchase')]
  partition 2: [('user-4', 'click')]
  partition 3: [('user-1', 'click'), ('user-2', 'click'), ('user-1', 'purchase'), ('user-1', 'logout'), ('user-2', 'purchase'), ('user-1', 'click'), ('user-2', 'logout')]


In [3]:
for p, messages in enumerate(log):
    keys_in_partition = set(k for k, _ in messages)
    print(f"  partition {p} keys: {keys_in_partition}")

assert all(
    len(set(partition_for_key(k) for k in set(k for k, _ in log[p]))) <= 1
    for p in range(NUM_PARTITIONS)
), "a key must map to exactly one partition"
print("  check: every key's events land in exactly one partition (order preserved per key). OK")


  partition 0 keys: set()
  partition 1 keys: {'user-3'}
  partition 2 keys: {'user-4'}
  partition 3 keys: {'user-2', 'user-1'}
  check: every key's events land in exactly one partition (order preserved per key). OK


**Note the skew already visible here**: partition 0 got 0 of the 10 events, partition 3 got 7,
for only 4 distinct user keys. Hash partitioning only balances load if the key space itself is
balanced — this is the same skew failure mode as `01`/`02`'s shuffle skew, now applied to a
stream (see Failure modes).

Now two independent consumers read from the (hot) partition 3, each tracking its own offset:


In [4]:
consumer_a_offsets = [0] * NUM_PARTITIONS
consumer_b_offsets = [0] * NUM_PARTITIONS

def consume_one(consumer_name, offsets, partition):
    offset = offsets[partition]
    if offset >= len(log[partition]):
        return None
    message = log[partition][offset]
    offsets[partition] += 1
    return message

print("consumer A reads partition 3 twice, consumer B reads partition 3 once:")
print(" A:", consume_one("A", consumer_a_offsets, 3))
print(" A:", consume_one("A", consumer_a_offsets, 3))
print(" B:", consume_one("B", consumer_b_offsets, 3))
print("consumer A offsets:", consumer_a_offsets)
print("consumer B offsets:", consumer_b_offsets)


consumer A reads partition 3 twice, consumer B reads partition 3 once:
 A: ('user-1', 'click')
 A: ('user-2', 'click')
 B: ('user-1', 'click')
consumer A offsets: [0, 0, 0, 2]
consumer B offsets: [0, 0, 0, 1]


Consumer A ends at position 2 on partition 3, consumer B at position 1 — the same underlying log,
read independently, with neither consumer's progress affecting the other's. This is the
"consumer group" idea in miniature: multiple readers, each with their own durable position into
a shared, ordered log.


## Part 3 — Crash-and-resume from a committed offset

**Hypothesis (stated first).** If a consumer commits its offset *after* each successfully
processed message (not before), a fresh consumer object created after a crash can read that
committed offset and resume from exactly the next unprocessed message — producing zero duplicate
processing and zero skipped messages, **provided the offset store itself survives the crash**
(here: a plain `dict` at module scope, standing in for what Kafka durably persists on the broker,
outside any single consumer process).

**Setup.** `Consumer` objects read their starting position from a shared `committed_offsets`
dict, not from their own `__init__` argument — that's the point: the offset lives outside the
object that can crash.


In [5]:
partition_log = [f"event-{i}" for i in range(10)]

# the durable offset store -- survives a consumer crash because it is NOT part of the consumer
# object itself (a real system would persist this externally, e.g. Kafka's __consumer_offsets
# topic; here a module-level dict stands in for "outside the crashing process").
committed_offsets = {}

class Consumer:
    def __init__(self, name, partition_log, offset_store, group="group-1"):
        self.name = name
        self.partition_log = partition_log
        self.offset_store = offset_store
        self.group = group
        # resume from the last committed offset for this group, or 0 if none exists yet
        self.position = self.offset_store.get(self.group, 0)
        print(f"  [{self.name}] starting at offset {self.position} "
              f"(from offset store: {self.group!r} -> {self.offset_store.get(self.group)})")

    def run(self, max_messages=None, crash_after=None):
        processed = []
        count = 0
        while self.position < len(self.partition_log):
            if crash_after is not None and count == crash_after:
                print(f"  [{self.name}] *** CRASH *** after processing {count} messages "
                      f"this run (last committed offset: {self.offset_store.get(self.group)})")
                return processed
            message = self.partition_log[self.position]
            # process the message
            processed.append(message)
            print(f"  [{self.name}] processed {message!r} at offset {self.position}")
            self.position += 1
            # commit AFTER successful processing (at-least-once semantics: see Failure modes)
            self.offset_store[self.group] = self.position
            count += 1
            if max_messages is not None and count >= max_messages:
                break
        return processed


In [6]:
print("-- run 1: consumer_1 processes 4 messages, then crashes before committing a 5th --")
consumer_1 = Consumer("consumer_1", partition_log, committed_offsets)
run1_processed = consumer_1.run(crash_after=4)

print(f"\noffset store after crash: {committed_offsets}")
print(f"(consumer_1's in-memory position was {consumer_1.position}, "
      f"but that in-memory state is gone -- only the offset store survives)")


-- run 1: consumer_1 processes 4 messages, then crashes before committing a 5th --
  [consumer_1] starting at offset 0 (from offset store: 'group-1' -> None)
  [consumer_1] processed 'event-0' at offset 0
  [consumer_1] processed 'event-1' at offset 1
  [consumer_1] processed 'event-2' at offset 2
  [consumer_1] processed 'event-3' at offset 3
  [consumer_1] *** CRASH *** after processing 4 messages this run (last committed offset: 4)

offset store after crash: {'group-1': 4}
(consumer_1's in-memory position was 4, but that in-memory state is gone -- only the offset store survives)


In [7]:
print("-- run 2: a FRESH consumer object picks up where the offset store says to resume --")
consumer_2 = Consumer("consumer_2", partition_log, committed_offsets)
run2_processed = consumer_2.run()

print(f"\nrun 1 processed: {run1_processed}")
print(f"run 2 processed: {run2_processed}")
all_processed = run1_processed + run2_processed
print(f"combined:        {all_processed}")
print(f"original log:    {partition_log}")

assert all_processed == partition_log, "resume must reproduce the full log with no gaps/dupes"
assert len(set(all_processed)) == len(all_processed), "no message should be processed twice"
print("\nresult: every message processed exactly once across the crash, in order, with no gaps"
      " and no duplicates -- confirming the hypothesis, GIVEN that the offset store itself"
      " survived the crash (see Failure modes below for what happens when it doesn't).")


-- run 2: a FRESH consumer object picks up where the offset store says to resume --
  [consumer_2] starting at offset 4 (from offset store: 'group-1' -> 4)
  [consumer_2] processed 'event-4' at offset 4
  [consumer_2] processed 'event-5' at offset 5
  [consumer_2] processed 'event-6' at offset 6
  [consumer_2] processed 'event-7' at offset 7
  [consumer_2] processed 'event-8' at offset 8
  [consumer_2] processed 'event-9' at offset 9

run 1 processed: ['event-0', 'event-1', 'event-2', 'event-3']
run 2 processed: ['event-4', 'event-5', 'event-6', 'event-7', 'event-8', 'event-9']
combined:        ['event-0', 'event-1', 'event-2', 'event-3', 'event-4', 'event-5', 'event-6', 'event-7', 'event-8', 'event-9']
original log:    ['event-0', 'event-1', 'event-2', 'event-3', 'event-4', 'event-5', 'event-6', 'event-7', 'event-8', 'event-9']

result: every message processed exactly once across the crash, in order, with no gaps and no duplicates -- confirming the hypothesis, GIVEN that the offset st

**Interpretation.** The two asserts pass: `run1_processed + run2_processed` reconstructs the
original log exactly, with no gaps and no repeats. `consumer_2` never saw `consumer_1` — it is a
genuinely fresh object — and still resumed correctly, because the *offset*, not the *consumer*,
carried the durable state. This is the entire mechanism a real broker gives you for free.

**Limitation, made concrete below**: the hypothesis had a condition — "provided the offset store
itself survives the crash." That's doing real work. If the offset store is only in-memory in the
same process as the consumer, the crash takes it too.


In [8]:
class InMemoryOnlyConsumer:
    """Keeps its offset only as an instance attribute -- nowhere else."""
    def __init__(self, name, partition_log):
        self.name = name
        self.partition_log = partition_log
        self.position = 0  # always starts from 0: no external memory of prior progress

    def run(self, crash_after=None):
        processed = []
        count = 0
        while self.position < len(self.partition_log):
            if crash_after is not None and count == crash_after:
                print(f"  [{self.name}] *** CRASH *** (in-memory offset {self.position} is now "
                      f"gone -- process restarted from nothing)")
                return processed
            message = self.partition_log[self.position]
            processed.append(message)
            self.position += 1
            count += 1
        return processed

print("-- run 1: processes 4 messages, then the process crashes (offset never externalized) --")
bad_consumer_1 = InMemoryOnlyConsumer("bad_consumer_1", partition_log)
bad_run1 = bad_consumer_1.run(crash_after=4)
print(f"processed: {bad_run1}")

print("\n-- run 2: a fresh process has NO record of prior progress, restarts at offset 0 --")
bad_consumer_2 = InMemoryOnlyConsumer("bad_consumer_2", partition_log)
bad_run2 = bad_consumer_2.run()
print(f"processed: {bad_run2}")

duplicates = [m for m in bad_run1 if m in bad_run2]
print(f"\nmessages processed twice: {duplicates}")
print(f"({len(duplicates)} of {len(partition_log)} messages were reprocessed) -- this is the")
print(f"'losing offset tracking on crash' failure mode: without a durable, external offset")
print(f"store, a fresh consumer cannot tell what already happened.")


-- run 1: processes 4 messages, then the process crashes (offset never externalized) --
  [bad_consumer_1] *** CRASH *** (in-memory offset 4 is now gone -- process restarted from nothing)
processed: ['event-0', 'event-1', 'event-2', 'event-3']

-- run 2: a fresh process has NO record of prior progress, restarts at offset 0 --
processed: ['event-0', 'event-1', 'event-2', 'event-3', 'event-4', 'event-5', 'event-6', 'event-7', 'event-8', 'event-9']

messages processed twice: ['event-0', 'event-1', 'event-2', 'event-3']
(4 of 10 messages were reprocessed) -- this is the
'losing offset tracking on crash' failure mode: without a durable, external offset
store, a fresh consumer cannot tell what already happened.


## Summary

- **Backpressure** is real and measurable: a bounded queue forces a fast producer to block at
  exactly the consumer's pace (Part 1's ~0.3s `put()` blocks).
- **Partitioning by key** gives per-key ordering and parallel consumption, but is only balanced if
  the key distribution is — the same skew Part 2 exposed by accident is the same mechanism
  `01`/`02` already introduced for batch shuffles.
- **Offsets stored outside the consumer process** are what make crash-and-resume correct (Part 3);
  offsets stored only in-memory reproduce the duplicate-processing failure mode Part 4 measured
  directly.

This is exactly what Kafka automates and hardens: a durable, replicated, partitioned log with
broker-managed consumer-group offsets — see `04-kafka` (not yet built) for the practical,
production-grade version of everything demonstrated from scratch here.
